<!-- # START -->

# 12c Infer expression from H&E Patches for Treatment Response models

Uses foundation model embeddings from H&E patches prepped in 12a-TR-prep.ipynb

In [ ]:
import os 
# import glob
# import json
# import numpy as np
import pandas as pd
from utils import load_features_from_csv

In [ ]:
from datetime import date

# Returns the current local date
today = date.today()
print("Today date is: ", today)

In [ ]:
ext_root_path = 'F:/01-data' 
root_path = "../data"
feature_path = os.path.join(root_path,'06-treatment_response','img_feats')
expression_path = os.path.join(root_path,'06-treatment_response','inferred_expression')

In [ ]:
model_path =  os.path.join(root_path,'04-results_test','trained_models')
feat_files = ['imgfeats_traintest','imgfeats_validation']

In [ ]:
from gene_model import GeneModelStore

# Predict expression from per-gene regressors

## Use Calibration

In [ ]:
APPLY_CALIBRATION = True
model_folder = 'calibrated'

In [ ]:
store_cd45 = GeneModelStore(model_dir=os.path.join(model_path,model_folder,'cd45'))
store_cd45.load_all()  
print(len(store_cd45.genes))

store_panck = GeneModelStore(model_dir=os.path.join(model_path,model_folder,'panck'))
store_panck.load_all()
print(len(store_panck.genes))

In [ ]:
for feat_file in feat_files:
    print(feat_file)
    image_names, feature_matrix = load_features_from_csv(os.path.join(feature_path,f'{feat_file}.csv'))
    feature_matrix_df = pd.DataFrame(feature_matrix, index=image_names)
    assert feature_matrix.shape[0] == len(image_names)
    assert feature_matrix_df.shape[0] == feature_matrix.shape[0]
    print(f"\t{feature_matrix.shape[0]} rows")

    gene_preds_cd45 = store_cd45.predict_many(None, feature_matrix, row_names=image_names, apply_calibration=APPLY_CALIBRATION)
    gene_preds_cd45.to_csv(os.path.join(expression_path,f"expression_cd45_{model_folder}_{feat_file}.csv"))


    gene_preds_panck = store_panck.predict_many(None, feature_matrix, row_names=image_names, apply_calibration=APPLY_CALIBRATION)
    gene_preds_panck.to_csv(os.path.join(expression_path,f"expression_panck_{model_folder}_{feat_file}.csv"))


    print("\tfinished inference")
    expr = pd.concat([
        gene_preds_cd45.add_suffix("_CD45"),
        gene_preds_panck.add_suffix("_PanCK")
    ], axis=1)
    print("\tnow saving")


    # save expression
    expr.to_csv(os.path.join(expression_path,f"expression_all_{model_folder}_{feat_file}.csv"))

<!-- ## Uncalibrated -->

## Without Calibration

In [ ]:
APPLY_CALIBRATION = False
model_folder = 'uncalibrated'

In [ ]:
store_cd45 = GeneModelStore(model_dir=os.path.join(model_path,model_folder,'cd45'))
store_cd45.load_all()  
print(len(store_cd45.genes))

store_panck = GeneModelStore(model_dir=os.path.join(model_path,model_folder,'panck'))
store_panck.load_all()
print(len(store_panck.genes))

In [ ]:
for feat_file in feat_files:
    print(feat_file)
    image_names, feature_matrix = load_features_from_csv(os.path.join(feature_path,f'{feat_file}.csv'))
    feature_matrix_df = pd.DataFrame(feature_matrix, index=image_names)
    assert feature_matrix.shape[0] == len(image_names)
    assert feature_matrix_df.shape[0] == feature_matrix.shape[0]
    print(f"\t{feature_matrix.shape[0]} rows")

    gene_preds_cd45 = store_cd45.predict_many(None, feature_matrix, row_names=image_names, apply_calibration=APPLY_CALIBRATION)
    gene_preds_cd45.to_csv(os.path.join(expression_path,f"expression_cd45_{model_folder}_{feat_file}.csv"))

    gene_preds_panck = store_panck.predict_many(None, feature_matrix, row_names=image_names, apply_calibration=APPLY_CALIBRATION)
    gene_preds_panck.to_csv(os.path.join(expression_path,f"expression_panck_{model_folder}_{feat_file}.csv"))
    print("\tfinished inference")

    expr = pd.concat([
        gene_preds_cd45.add_suffix("_CD45"),
        gene_preds_panck.add_suffix("_PanCK")
    ], axis=1)
    print("\tnow saving")

    # save expression
    expr.to_csv(os.path.join(expression_path,f"expression_all_{model_folder}_{feat_file}.csv"))

<!-- # Evaluate expression cohorts -->

# END

<!-- # Repair LEAP003
accidentally included the expression for the resection -->

<!-- Fix image features too -->